In [640]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [641]:
#prepare data
def prepare_data(df):
    # set all column name to lower case
    df.columns = df.columns.str.lower()
    # check if it contains date and close price column, if not, report error and exit the function
    if not 'date' in df.columns or not 'close' in df.columns:
        raise ValueError('DataFrame must have at least contain date and close price column')
    # rename the column name
    df = df.rename(columns={
        'date': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    })
    # if there is no Open, High, Low, fill it with close price
    if not 'Open' in df.columns:
        df['Open'] = df['Close']
    if not 'High' in df.columns:
        df['High'] = df['Close']
    if not 'Low' in df.columns:
        df['Low'] = df['Close']
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')

In [642]:
# define a object called Strategy
class Strategy:
    def __init__(self):
        self.data = None
        self.position = 0
        self.orders = []
        self.indicators = {}
    def init(self):
        """Initialize strategy (pre-run)"""
        pass
    
    def next(self,i):
        """Main strategy logic"""
        pass
    
    def buy(self):
        self.orders.append(('buy', self.data.Close.iloc[-1]))
        self.position =1
        return self.position
    def sell(self):
        self.orders.append(('sell', self.data.Close.iloc[-1]))
        self.position =-1
        return self.position
    def I(self, func, *args, **kwargs):
        """添加指标计算"""
        indicator_name = f'{func.__name__}_{args}_{kwargs}'
        if indicator_name not in self.indicators:
            self.indicators[indicator_name] = func( *args)
        return self.indicators[indicator_name]

    # 添加技术指标函数
def SMA(series, window):
        return series.rolling(window).mean()
def crossover(a, b):
        """判断两个序列是否发生交叉"""
        return a.iloc[-2] < b.iloc[-2] and a.iloc[-1] > b.iloc[-1]


In [643]:
def backtest(df, strategy, commission=0.002,  initial_cash=100000):
    strategy_instance = strategy()
    strategy_instance.data = df
    strategy_instance.init()
    slippage = 0.0009  # 假设的滑点
    cash = initial_cash
    holdings = 0
    equity = [initial_cash]
    positions = []
    trades = []  # 存储每笔交易信息
    current_trade = None  # 当前未平仓的交易
    
    for i in range(len(df)):
        current_date = df.index[i]
        current_price = df.iloc[i]['Close']
        
        # 调用策略获取信号
        signal = strategy_instance.next(i+1)
        
        # 保存当前持仓方向
        current_position = 1 if holdings > 0 else (-1 if holdings < 0 else 0)
        positions.append(current_position)
        
        # 执行交易逻辑
        if signal is not None and signal != current_position:
            # 先平仓
            if holdings > 0 and signal <= 0:  # 平多仓
                close_price = current_price * (1 - slippage)
                cash += holdings * close_price * (1 - commission)
                
                # 记录交易结束
                if current_trade:
                    current_trade['exit_date'] = current_date
                    current_trade['exit_price'] = close_price
                    current_trade['return_pct'] = (close_price / current_trade['entry_price'] - 1) * 100
                    trades.append(current_trade)
                    current_trade = None
                
                holdings = 0
                
            elif holdings < 0 and signal >= 0:  # 平空仓
                close_price = current_price * (1 + slippage)
                cash += holdings * close_price * (1 - commission)
                
                # 记录交易结束
                if current_trade:
                    current_trade['exit_date'] = current_date
                    current_trade['exit_price'] = close_price
                    current_trade['return_pct'] = (current_trade['entry_price'] / close_price - 1) * 100
                    trades.append(current_trade)
                    current_trade = None
                
                holdings = 0
            
            # 再开新仓
            if signal == 1:  # 开多仓
                buy_price = current_price * (1 + slippage)
                cost_per_share = buy_price * (1 + commission)
                shares = cash / cost_per_share
                holdings = shares
                cash = 0
                
                # 记录新交易
                current_trade = {
                    'entry_date': current_date,
                    'entry_price': buy_price,
                    'direction': 1,
                    'size': shares
                }
                
            elif signal == -1:  # 开空仓
                sell_price = current_price * (1 - slippage)
                proceeds_per_share = sell_price * (1 - commission)
                shares = cash / (sell_price * (1 + commission))
                holdings = -shares
                cash += shares * proceeds_per_share
                
                # 记录新交易
                current_trade = {
                    'entry_date': current_date,
                    'entry_price': sell_price,
                    'direction': -1,
                    'size': shares
                }
        
        # 计算当日权益（现金 + 持仓市值）
        position_value = holdings * current_price
        current_equity = cash + position_value
        equity.append(current_equity)
    
    # 处理最后未平仓的交易
    if current_trade:
        current_trade['exit_date'] = df.index[-1]
        current_trade['exit_price'] = df.iloc[-1]['Close']
        if current_trade['direction'] == 1:
            current_trade['return_pct'] = (current_trade['exit_price'] / current_trade['entry_price'] - 1) * 100
        else:
            current_trade['return_pct'] = (current_trade['entry_price'] / current_trade['exit_price'] - 1) * 100
        trades.append(current_trade)
    
    # 准备计算指标的数据
    equity_series = pd.Series(equity[1:], index=df.index)
    returns = equity_series.pct_change().fillna(0)
    n_days = len(df) #交易日总数
    n_years = n_days / 240 #年数
    
    # 计算基准（买入持有）收益率
    bh_returns = df['Close'].pct_change().fillna(0)
    bh_final_return = (df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100
    
    # 计算回撤
    rolling_max = equity_series.expanding(min_periods=1).max()
    drawdown = (equity_series - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100
    avg_drawdown = drawdown.mean() * 100
    
    # 计算交易指标
    trades_df = pd.DataFrame(trades)
    n_trades = len(trades_df)
    
    if n_trades > 0:
        win_rate = (trades_df['return_pct'] > 0).mean() * 100
        best_trade = trades_df['return_pct'].max()
        worst_trade = trades_df['return_pct'].min()
        avg_trade = trades_df['return_pct'].mean()
        
        # 计算交易持续时间
        trades_df['duration'] = (trades_df['exit_date'] - trades_df['entry_date']).dt.days
        max_trade_duration = trades_df['duration'].max()
        avg_trade_duration = trades_df['duration'].mean()
        
        # 盈利因子
        winning_trades = trades_df[trades_df['return_pct'] > 0]
        losing_trades = trades_df[trades_df['return_pct'] <= 0]
        profit_factor = abs(winning_trades['return_pct'].sum() / losing_trades['return_pct'].sum()) if len(losing_trades) > 0 else np.inf
        
        # 期望收益
        expectancy = (winning_trades['return_pct'].mean() * (len(winning_trades)/n_trades) + 
                     losing_trades['return_pct'].mean() * (len(losing_trades)/n_trades))
        
        # SQN (System Quality Number)
        avg_profit_per_trade = trades_df['return_pct'].mean()
        std_profit_per_trade = trades_df['return_pct'].std()
        sqn = (avg_profit_per_trade / std_profit_per_trade) * np.sqrt(n_trades) if std_profit_per_trade != 0 else 0
        
        # 凯利准则
        win_prob = len(winning_trades) / n_trades
        win_loss_ratio = abs(winning_trades['return_pct'].mean() / losing_trades['return_pct'].mean()) if len(losing_trades) > 0 else np.inf
        kelly = win_prob - (1 - win_prob) / win_loss_ratio if win_loss_ratio != np.inf else win_prob
    else:
        win_rate = best_trade = worst_trade = avg_trade = 0
        max_trade_duration = avg_trade_duration = 0
        profit_factor = expectancy = sqn = kelly = 0
    
    # 计算风险指标
    # 年化收益率
    ann_return = ((equity_series.iloc[-1] / initial_cash) ** (1/n_years) - 1) if n_years > 0 else 0
    
    # 年化波动率
    ann_volatility = returns.std() * np.sqrt(240) * 100
    
    # CAGR
    cagr = ((equity_series.iloc[-1] / initial_cash) ** (1/n_years) - 1) * 100 if n_years > 0 else 0
    
    # 夏普比率
    sharpe_ratio = (ann_return *100 / ann_volatility) if ann_volatility != 0 else 0
    
    # 索提诺比率
    downside_returns = returns[returns < 0]
    downside_volatility = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 1 else 0
    sortino_ratio = (ann_return / downside_volatility) if downside_volatility != 0 else 0
    
    # 卡玛比率
    calmar_ratio = (ann_return / abs(max_drawdown/100)) if max_drawdown != 0 else 0
    
    # Alpha 和 Beta
    if len(returns) > 1 and len(bh_returns) > 1:
        covariance = np.cov(returns[1:], bh_returns[1:len(returns)])[0, 1]
        variance = np.var(bh_returns[1:len(returns)])
        beta = covariance / variance if variance != 0 else 0
        alpha = (ann_return - 0.02) - beta * (bh_final_return/n_years - 0.02)  # 假设无风险利率为2%
    else:
        alpha = beta = 0
    
    # 计算持仓时间比例
    exposure_time = (pd.Series(positions) != 0).mean() * 100
    
    # 计算总佣金
    total_commission = 0
    for trade in trades:
        entry_commission = trade['entry_price'] * trade['size'] * commission
        exit_commission = trade['exit_price'] * trade['size'] * commission
        total_commission += entry_commission + exit_commission
    
    # 返回所有统计指标
    stats = {
        'Exposure Time [%]': exposure_time,
        'Equity Final [$]': equity_series.iloc[-1],
        'Equity Final [%]': (equity_series.iloc[-1] / initial_cash ) * 100,
        'Equity Peak [$]': equity_series.max(),
        'Commissions [$]': total_commission,
        'Return [%]': (equity_series.iloc[-1] / initial_cash - 1) * 100,
        'Buy & Hold Return [%]': bh_final_return,
        'Return (Ann.) [%]': ann_return * 100,
        'Volatility (Ann.) [%]': ann_volatility,
        'CAGR [%]': cagr,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Alpha [%]': alpha * 100,
        'Beta': beta,
        'Max. Drawdown [%]': max_drawdown,
        'Avg. Drawdown [%]': avg_drawdown,
        '# Trades': n_trades,
        'Win Rate [%]': win_rate,
        'Best Trade [%]': best_trade,
        'Worst Trade [%]': worst_trade,
        'Avg. Trade [%]': avg_trade,
        'Max. Trade Duration': f"{int(max_trade_duration)} days" if max_trade_duration > 0 else "0 days",
        'Avg. Trade Duration': f"{int(avg_trade_duration)} days" if avg_trade_duration > 0 else "0 days",
        'Profit Factor': profit_factor,
        'Expectancy [%]': expectancy,
        'SQN': sqn,
        'Kelly Criterion': kelly,
        '_equity_curve': equity_series,
        '_trades': trades_df
    }
    
    return stats

In [644]:
# 纯双均值策略（0）
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self,i):
        if i < 20:
            return
        if crossover(self.ma1[:i], self.ma2[:i]):
            return self.buy()
        elif crossover(self.ma2[:i], self.ma1[:i]):
            return self.sell()

In [645]:
bu = pd.read_csv('data/bu.csv')
jd = pd.read_csv('data/jd.csv')
l = pd.read_csv('data/l.csv')
pp = pd.read_csv('data/pp.csv')
ru = pd.read_csv('data/ru.csv')
v = pd.read_csv('data/v.csv')
bu = prepare_data(bu)
jd = prepare_data(jd)
l = prepare_data(l)
pp = prepare_data(pp)
ru = prepare_data(ru)
v = prepare_data(v)
backtest(bu,SmaCross)

{'Exposure Time [%]': np.float64(98.93092105263158),
 'Equity Final [$]': np.float64(33114.29896507815),
 'Equity Final [%]': np.float64(33.114298965078156),
 'Equity Peak [$]': np.float64(105963.45167692078),
 'Commissions [$]': np.float64(31311.838703109857),
 'Return [%]': np.float64(-66.88570103492184),
 'Buy & Hold Return [%]': np.float64(20.371621621621628),
 'Return (Ann.) [%]': np.float64(-10.332901877239664),
 'Volatility (Ann.) [%]': np.float64(29.313748173735064),
 'CAGR [%]': np.float64(-10.332901877239664),
 'Sharpe Ratio': np.float64(-0.3524933698686093),
 'Sortino Ratio': np.float64(-0.4957051895548897),
 'Calmar Ratio': np.float64(-0.15029824067854347),
 'Alpha [%]': np.float64(-29.45386452511864),
 'Beta': np.float64(0.0860195393792005),
 'Max. Drawdown [%]': np.float64(-68.74932022218132),
 'Avg. Drawdown [%]': np.float64(-42.56939491242525),
 '# Trades': 131,
 'Win Rate [%]': np.float64(35.11450381679389),
 'Best Trade [%]': np.float64(43.4707098794215),
 'Worst Trad

In [646]:
backtest(jd,SmaCross,)

{'Exposure Time [%]': np.float64(98.64309210526315),
 'Equity Final [$]': np.float64(15792.704110048257),
 'Equity Final [%]': np.float64(15.792704110048255),
 'Equity Peak [$]': np.float64(118416.55454441758),
 'Commissions [$]': np.float64(24317.804099113706),
 'Return [%]': np.float64(-84.20729588995175),
 'Buy & Hold Return [%]': np.float64(-13.096695226438193),
 'Return (Ann.) [%]': np.float64(-16.6510155996067),
 'Volatility (Ann.) [%]': np.float64(33.73625024312747),
 'CAGR [%]': np.float64(-16.6510155996067),
 'Sharpe Ratio': np.float64(-0.49356450345274333),
 'Sortino Ratio': np.float64(-0.5113812973595696),
 'Calmar Ratio': np.float64(-0.19162755420473032),
 'Alpha [%]': np.float64(-32.80660838176151),
 'Beta': np.float64(-0.10785731025702158),
 'Max. Drawdown [%]': np.float64(-86.89259573713053),
 'Avg. Drawdown [%]': np.float64(-57.15120646563827),
 '# Trades': 126,
 'Win Rate [%]': np.float64(34.12698412698413),
 'Best Trade [%]': np.float64(23.405959691780097),
 'Worst Tr

In [647]:
backtest(l,SmaCross)

{'Exposure Time [%]': np.float64(97.81982723159194),
 'Equity Final [$]': np.float64(66677.11142629127),
 'Equity Final [%]': np.float64(66.67711142629128),
 'Equity Peak [$]': np.float64(122404.882142927),
 'Commissions [$]': np.float64(48539.282422813216),
 'Return [%]': np.float64(-33.32288857370873),
 'Buy & Hold Return [%]': np.float64(-26.038500506585617),
 'Return (Ann.) [%]': np.float64(-3.922400933812742),
 'Volatility (Ann.) [%]': np.float64(18.168857563251525),
 'CAGR [%]': np.float64(-3.922400933812742),
 'Sharpe Ratio': np.float64(-0.21588594220399532),
 'Sortino Ratio': np.float64(-0.3100495790529392),
 'Calmar Ratio': np.float64(-0.08504495657998243),
 'Alpha [%]': np.float64(-9.968081497693431),
 'Beta': np.float64(-0.015616493953386967),
 'Max. Drawdown [%]': np.float64(-46.12149963441786),
 'Avg. Drawdown [%]': np.float64(-20.97345046906436),
 '# Trades': 129,
 'Win Rate [%]': np.float64(37.2093023255814),
 'Best Trade [%]': np.float64(11.002845080936584),
 'Worst Tra

In [648]:
backtest(pp,SmaCross)

{'Exposure Time [%]': np.float64(98.31345125462772),
 'Equity Final [$]': np.float64(50714.84537996073),
 'Equity Final [%]': np.float64(50.71484537996073),
 'Equity Peak [$]': np.float64(164571.81288293),
 'Commissions [$]': np.float64(53310.3530252552),
 'Return [%]': np.float64(-49.28515462003927),
 'Buy & Hold Return [%]': np.float64(-18.373909049150207),
 'Return (Ann.) [%]': np.float64(-6.483225106439305),
 'Volatility (Ann.) [%]': np.float64(19.92131454318633),
 'CAGR [%]': np.float64(-6.483225106439305),
 'Sharpe Ratio': np.float64(-0.32544163149397975),
 'Sortino Ratio': np.float64(-0.44931830863104255),
 'Calmar Ratio': np.float64(-0.09351097982358687),
 'Alpha [%]': np.float64(-3.306205220592461),
 'Beta': np.float64(0.028228632284659345),
 'Max. Drawdown [%]': np.float64(-69.3311643046649),
 'Avg. Drawdown [%]': np.float64(-33.42799229984442),
 '# Trades': 133,
 'Win Rate [%]': np.float64(33.08270676691729),
 'Best Trade [%]': np.float64(26.274503814390826),
 'Worst Trade [

In [649]:
backtest(ru,SmaCross)

{'Exposure Time [%]': np.float64(98.84868421052632),
 'Equity Final [$]': np.float64(162879.6793681762),
 'Equity Final [%]': np.float64(162.8796793681762),
 'Equity Peak [$]': np.float64(255521.5252924567),
 'Commissions [$]': np.float64(86359.05813273386),
 'Return [%]': np.float64(62.8796793681762),
 'Buy & Hold Return [%]': np.float64(4.932735426008961),
 'Return (Ann.) [%]': np.float64(4.931992208311575),
 'Volatility (Ann.) [%]': np.float64(28.830345416603926),
 'CAGR [%]': np.float64(4.931992208311575),
 'Sharpe Ratio': np.float64(0.17106948033550615),
 'Sortino Ratio': np.float64(0.2561488364279136),
 'Calmar Ratio': np.float64(0.10296773205611172),
 'Alpha [%]': np.float64(-4.074737256389888),
 'Beta': np.float64(0.15010675077835606),
 'Max. Drawdown [%]': np.float64(-47.89842516511787),
 'Avg. Drawdown [%]': np.float64(-21.47715354299767),
 '# Trades': 130,
 'Win Rate [%]': np.float64(46.92307692307692),
 'Best Trade [%]': np.float64(47.417049714173196),
 'Worst Trade [%]': n

In [650]:
backtest(v,SmaCross)

{'Exposure Time [%]': np.float64(90.08264462809917),
 'Equity Final [$]': np.float64(86306.10077245123),
 'Equity Final [%]': np.float64(86.30610077245123),
 'Equity Peak [$]': np.float64(115603.89775205735),
 'Commissions [$]': np.float64(4670.826975892055),
 'Return [%]': np.float64(-13.693899227548767),
 'Buy & Hold Return [%]': np.float64(-29.824561403508774),
 'Return (Ann.) [%]': np.float64(-13.588791552612744),
 'Volatility (Ann.) [%]': np.float64(26.16023119497393),
 'CAGR [%]': np.float64(-13.588791552612744),
 'Sharpe Ratio': np.float64(-0.5194446276615288),
 'Sortino Ratio': np.float64(-0.8604276601365297),
 'Calmar Ratio': np.float64(-0.5361895538820761),
 'Alpha [%]': np.float64(-553.5604481441894),
 'Beta': np.float64(-0.18175898686506312),
 'Max. Drawdown [%]': np.float64(-25.34326052088907),
 'Avg. Drawdown [%]': np.float64(-9.88694592894346),
 '# Trades': 12,
 'Win Rate [%]': np.float64(41.66666666666667),
 'Best Trade [%]': np.float64(13.330317054770834),
 'Worst Trad